In [ ]:
import dspy
llama32 = dspy.LM('ollama_chat/llama3.2', api_base='http://localhost:11434', api_key='')
gpt_oss = dspy.LM('ollama_chat/gpt-oss:latest', api_base='http://localhost:11434', api_key='')

# Signatures

## Inline DSPy Signatures

In [ ]:
with dspy.context(lm=llama32):
    toxicity = dspy.Predict(
        dspy.Signature(
            "comment -> toxic: bool",
            instructions="Mark as 'toxic' if the comment includes insults, harassment, or sarcastic derogatory remarks.",
        )
    )
    comment = "you are beautiful."
    output = toxicity(comment=comment).toxic
    print(output)

False


### Example A: Sentiment Classification

In [ ]:
with dspy.context(lm=llama32):
    sentence = "it's a charming and often affecting journey."  # example from the SST-2 dataset.

    classify = dspy.Predict('sentence -> sentiment: bool')  # we'll see an example with Literal[] later
    output = classify(sentence=sentence).sentiment
    print(output)

True


### Example B: Summarization

In [15]:
with dspy.context(lm=llama32):
    # Example from the XSum dataset.
    document = """The 21-year-old made seven appearances for the Hammers and netted his only goal for them in a Europa League qualification round match against Andorran side FC Lustrains last season. Lee had two loan spells in League One last term, with Blackpool and then Colchester United. He scored twice for the U's but was unable to save them from relegation. The length of Lee's contract with the promoted Tykes has not been revealed. Find all the latest football transfers on our dedicated page."""

    summarize = dspy.ChainOfThought('document -> summary')
    response = summarize(document=document)

    print(response.summary)
    print("Reasoning:", response.reasoning)

Lee is a 21-year-old football player who has made appearances for several clubs, including Hammers and had loan spells at Blackpool and Colchester United. The document mentions his goal for Hammers in a Europa League qualification round match and his loan spell at Bradford City, but the contract details are unknown.
Reasoning: The document is about a 21-year-old football player named Lee, who made appearances for Hammers, Blackpool, and Colchester United, but the main focus is on his loan spells and contract with Bradford City.


## Class-based DSPy Signatures

### Example C: Classification

In [16]:
from typing import Literal

class Emotion(dspy.Signature):
    """Classify emotion."""

    sentence: str = dspy.InputField()
    sentiment: Literal['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'] = dspy.OutputField()

with dspy.context(lm=llama32):
    sentence = "i started feeling a little vulnerable when the giant spotlight started blinding me"  # from dair-ai/emotion

    classify = dspy.Predict(Emotion)
    output = classify(sentence=sentence)
    print(output)

Prediction(
    sentiment='fear'
)


### Example D: A metric that evaluates faithfulness to citations

In [18]:
class CheckCitationFaithfulness(dspy.Signature):
    """Verify that the text is based on the provided context."""

    context: str = dspy.InputField(desc="facts here are assumed to be true")
    text: str = dspy.InputField()
    faithfulness: bool = dspy.OutputField()
    evidence: dict[str, list[str]] = dspy.OutputField(desc="Supporting evidence for claims")

with dspy.context(lm=llama32):
    context = "The 21-year-old made seven appearances for the Hammers and netted his only goal for them in a Europa League qualification round match against Andorran side FC Lustrains last season. Lee had two loan spells in League One last term, with Blackpool and then Colchester United. He scored twice for the U's but was unable to save them from relegation. The length of Lee's contract with the promoted Tykes has not been revealed. Find all the latest football transfers on our dedicated page."

    text = "Lee scored 3 goals for Colchester United."

    faithfulness = dspy.ChainOfThought(CheckCitationFaithfulness)
    output = faithfulness(context=context, text=text)
    print(output)

Prediction(
    reasoning='This statement is based on the provided context.',
    faithfulness=False,
    evidence={'claims': ['Lee scored 3 goals for Colchester United.'], 'evidence_type': []}
)


### Example E: Multi-modal image classification

In [20]:
class DogPictureSignature(dspy.Signature):
    """Output the dog breed of the dog in the image."""
    image_1: dspy.Image = dspy.InputField(desc="An image of a dog")
    answer: str = dspy.OutputField(desc="The dog breed of the dog in the image")

with dspy.context(lm=gpt_oss):
    image_url = "https://picsum.photos/id/237/200/300"
    classify = dspy.Predict(DogPictureSignature)
    output = classify(image_1=dspy.Image.from_url(image_url))
    print(output)

/var/folders/g0/x5c5f2j50qs15lysg89cbmvh0000gn/T/ipykernel_12511/169671538.py:9: DeprecationWarning: Image.from_url is deprecated; use Image(url) instead.
  output = classify(image_1=dspy.Image.from_url(image_url))


APIConnectionError: litellm.APIConnectionError: Ollama_chatException - {"error":"illegal base64 data at input byte 5"}

## Types Resolution in Signatures

### Working with Custom Types

In [21]:
# Simple custom type
import pydantic


class QueryResult(pydantic.BaseModel):
    text: str
    score: float

signature = dspy.Signature("query: str -> result: QueryResult")

class MyContainer:
    class Query(pydantic.BaseModel):
        text: str
    class Score(pydantic.BaseModel):
        score: float

signature = dspy.Signature("query: MyContainer.Query -> score: MyContainer.Score")